# Lecture 10 — DMRG II: TeNPy and Applications

---

## Overview

Lecture 09 described the DMRG algorithm from first principles. This lecture puts it into practice using **TeNPy** (Tensor Network Python), a well-maintained library that handles the low-level implementation. The focus here is on the workflow: how to define a model, run DMRG, interpret the output, and use the results to explore the TFIM phase diagram at system sizes far beyond ED reach.

---

In [ ]:
import numpy as np
import scipy.sparse as sp
from scipy.sparse.linalg import eigsh
from scipy.optimize import curve_fit
import matplotlib.pyplot as plt

plt.rcParams.update({'font.size': 12, 'figure.dpi': 100})

# Check TeNPy availability
try:
    import tenpy
    from tenpy.models.tf_ising import TFIChain
    from tenpy.networks.mps import MPS
    from tenpy.algorithms import dmrg
    TENPY_AVAILABLE = True
    print(f"TeNPy version: {tenpy.__version__}")
except ImportError:
    TENPY_AVAILABLE = False
    print("TeNPy not found. Install with: pip install physics-tenpy")
    print("Cells requiring TeNPy will be skipped gracefully.")


# ED reference (reused throughout for benchmarking)
def build_tfim_ed(N, J=1.0, Gamma=1.0):
    dim = 2**N
    rows, cols, data = [], [], []
    for state in range(dim):
        diag = 0.0
        for i in range(N - 1):
            sz_i = 1 if (state >> i) & 1 else -1
            sz_j = 1 if (state >> (i + 1)) & 1 else -1
            diag -= J * sz_i * sz_j
        if diag != 0.0:
            rows.append(state); cols.append(state); data.append(diag)
        for i in range(N):
            rows.append(state); cols.append(state ^ (1 << i)); data.append(-Gamma)
    return sp.csr_matrix((data, (rows, cols)), shape=(dim, dim))

## 1. TeNPy: Setup and Interface

TeNPy (Tensor Network Python) is a Python library developed primarily by Johannes Hauschild and Frank Pollmann for tensor network algorithms in condensed matter physics. It provides:

- Implementations of DMRG (finite and infinite), TEBD, VUMPS, and other algorithms.
- A model library with many standard Hamiltonians (Heisenberg, TFIM, Hubbard, etc.) and the ability to define custom models.
- Automatic handling of Abelian symmetries (e.g., conserved total $S^z$), which dramatically reduces the effective bond dimension and computational cost.
- Tools for computing observables, correlation functions, and entanglement spectra.

Installation:

```bash
pip install physics-tenpy
```

A minimal DMRG run has four steps: (1) define the model, (2) define the MPS, (3) configure the DMRG engine, (4) run. TeNPy's API makes each step explicit, and the output is a converged MPS object from which any observable can be computed.

---

## 2. Defining Models in TeNPy

TeNPy models are defined by specifying the Hamiltonian in terms of local operators and couplings. For the 1D TFIM:

$$\hat{H} = -J \sum_i \hat{\sigma}^z_i \hat{\sigma}^z_{i+1} - g \sum_i \hat{\sigma}^x_i$$

TeNPy's built-in `TFIChain` model implements this directly. Custom models are defined by subclassing `CouplingMPOModel` and specifying the bond terms via `add_coupling`.

A key feature: TeNPy automatically constructs the Hamiltonian as a **Matrix Product Operator (MPO)** — the operator analogue of an MPS. This MPO representation is what allows the effective Hamiltonian $H_{\text{eff}}$ to be computed efficiently during DMRG.

---

In [ ]:
if TENPY_AVAILABLE:
    # Define the TFIM model at the critical point
    model_params = {
        'J': 1.0,           # Ising coupling
        'g': 1.0,           # transverse field strength (Gamma/J = g)
        'L': 20,            # chain length
        'bc_MPS': 'finite'  # open boundary conditions
    }
    model = TFIChain(model_params)
    print("Model defined successfully.")
    print(f"  L = {model_params['L']}, J = {model_params['J']}, g = {model_params['g']}")
    print(f"  Hilbert space dimension: 2^{model_params['L']} = {2**model_params['L']}")
else:
    print("Skipping: TeNPy not available.")

## 3. Running DMRG and Reading the Output

Once the model is defined, DMRG is run by:
1. Initialising an MPS (random or product state)
2. Setting DMRG parameters (bond dimension, truncation threshold, mixer settings)
3. Running the engine sweep by sweep

At each sweep, TeNPy reports:
- **Energy per site:** converges from above to the true ground state energy
- **Maximum discarded weight:** should decrease as sweeps continue
- **Entropy at each bond:** the entanglement entropy profile

The `mixer` option adds small noise to the MPS during early sweeps to prevent DMRG from getting trapped in local minima — particularly important in the ordered phase where the two symmetry-related ground states can mix.

---

In [ ]:
if TENPY_AVAILABLE:
    import logging
    logging.disable(logging.WARNING)  # suppress verbose TeNPy output

    model_params = {'J': 1.0, 'g': 1.0, 'L': 20, 'bc_MPS': 'finite'}
    model = TFIChain(model_params)

    # Initialise from a product state
    psi0 = MPS.from_lat_product_state(model.lat, p_state=['up', 'down'])

    dmrg_params = {
        'trunc_params': {'chi_max': 64, 'svd_min': 1e-10},
        'mixer': True,
        'mixer_params': {'amplitude': 1e-5, 'decay': 1.5, 'disable_after': 50},
        'N_sweeps_check': 1,
        'min_sweeps': 3,
        'max_sweeps': 20,
    }

    engine = dmrg.TwoSiteDMRGEngine(psi0, model, dmrg_params)
    E0_tenpy, psi_gs = engine.run()

    print(f"DMRG energy (L=20, g=1.0):  E0 = {E0_tenpy:.8f}")
    print(f"DMRG energy per site:        E0/L = {E0_tenpy/20:.8f}")

    # Compare with ED
    H_ed = build_tfim_ed(20, J=1.0, Gamma=1.0)
    E0_ed = eigsh(H_ed, k=1, which='SA', return_eigenvectors=False)[0]
    print(f"ED energy (L=20, g=1.0):    E0 = {E0_ed:.8f}")
    print(f"Relative difference: {abs(E0_tenpy - E0_ed)/abs(E0_ed):.2e}")
else:
    print("Skipping: TeNPy not available.")

## 4. DMRG vs ED: A Comparison

| Property | Exact Diagonalisation | DMRG |
|---|---|---|
| System sizes | Up to ~20–22 sites | Up to thousands of sites |
| Cost | $O(4^N)$ memory | $O(N \chi^3)$ time |
| Accuracy | Exact | Controlled by $\chi$ |
| Phase diagram | All parameters at once | One point at a time |
| Excited states | Full spectrum | First few states |
| Dimensions | Any | 1D (or narrow 2D) |

ED and DMRG are **complementary**: use ED for small systems to validate, then use DMRG to access the thermodynamic limit. The cell below validates DMRG at several sizes against ED — this should be your first step whenever starting a new DMRG calculation.

---

In [ ]:
if TENPY_AVAILABLE:
    import logging
    logging.disable(logging.WARNING)

    sizes = [8, 12, 16, 20]
    g_values = [0.5, 1.0, 2.0]

    print(f"{'L':>4}  {'g':>5}  {'E0 (ED)':>14}  {'E0 (DMRG)':>14}  {'Rel. error':>12}")
    print("-" * 60)

    for N in sizes:
        for g in g_values:
            # ED
            H_ed = build_tfim_ed(N, J=1.0, Gamma=g)
            E0_ed = eigsh(H_ed, k=1, which='SA', return_eigenvectors=False)[0]

            # DMRG
            model = TFIChain({'J': 1.0, 'g': g, 'L': N, 'bc_MPS': 'finite'})
            psi = MPS.from_lat_product_state(model.lat, p_state=['up', 'down'])
            params = {
                'trunc_params': {'chi_max': 64, 'svd_min': 1e-12},
                'mixer': True,
                'mixer_params': {'amplitude': 1e-5, 'decay': 1.5, 'disable_after': 30},
                'N_sweeps_check': 1, 'min_sweeps': 3, 'max_sweeps': 20,
            }
            engine = dmrg.TwoSiteDMRGEngine(psi, model, params)
            E0_dmrg, _ = engine.run()

            rel_err = abs(E0_dmrg - E0_ed) / abs(E0_ed)
            print(f"{N:>4}  {g:>5.1f}  {E0_ed:>14.8f}  {E0_dmrg:>14.8f}  {rel_err:>12.2e}")
else:
    # Show expected output structure without TeNPy
    print("TeNPy not available — showing ED energies only.")
    sizes = [8, 12, 16, 20]
    g_values = [0.5, 1.0, 2.0]
    print(f"{'L':>4}  {'g':>5}  {'E0 (ED)':>14}")
    print("-" * 30)
    for N in sizes:
        for g in g_values:
            H = build_tfim_ed(N, J=1.0, Gamma=g)
            E0 = eigsh(H, k=1, which='SA', return_eigenvectors=False)[0]
            print(f"{N:>4}  {g:>5.1f}  {E0:>14.8f}")

## 5. Exploring the TFIM Phase Diagram with DMRG

Once DMRG is validated against ED, the power comes from scanning $\Gamma/J$ for system sizes far beyond ED reach.

**Phase boundary:** Run DMRG at a range of $\Gamma/J$ values for several system sizes $L$. Extract the energy gap $\Delta = E_1 - E_0$ and plot $L \cdot \Delta$ vs $\Gamma/J$ — the crossing identifies the critical point.

**Order parameter:** In the ordered phase with a small symmetry-breaking field, $\langle \hat{\sigma}^z_{L/2} \rangle$ is non-zero; it vanishes in the disordered phase.

At $L = 200$–$500$, finite-size corrections are $\lesssim 1\%$, giving a much cleaner determination of the critical point and exponents than ED at $L = 8$–$20$.

---

In [ ]:
if TENPY_AVAILABLE:
    import logging
    logging.disable(logging.WARNING)

    # Phase diagram scan: gap and order parameter vs Gamma/J
    g_scan = np.linspace(0.4, 1.6, 13)
    sizes_scan = [40, 80, 120]
    chi_scan = 64

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    colors = ['tab:blue', 'tab:orange', 'tab:green']

    for L, color in zip(sizes_scan, colors):
        gaps = []
        for g in g_scan:
            model = TFIChain({'J': 1.0, 'g': g, 'L': L, 'bc_MPS': 'finite'})

            # Ground state
            psi0 = MPS.from_lat_product_state(model.lat, p_state=['up', 'down'])
            params0 = {
                'trunc_params': {'chi_max': chi_scan, 'svd_min': 1e-10},
                'mixer': True,
                'mixer_params': {'amplitude': 1e-5, 'decay': 1.5, 'disable_after': 30},
                'N_sweeps_check': 1, 'min_sweeps': 3, 'max_sweeps': 15,
            }
            E0, psi0_gs = dmrg.TwoSiteDMRGEngine(psi0, model, params0).run()

            # First excited state (orthogonal to ground state)
            psi1 = MPS.from_lat_product_state(model.lat, p_state=['up', 'up'])
            params1 = dict(params0)
            params1['orthogonal_to'] = [psi0_gs]
            E1, _ = dmrg.TwoSiteDMRGEngine(psi1, model, params1).run()

            gaps.append(E1 - E0)

        scaled_gap = np.array(gaps) * L
        axes[0].plot(g_scan, scaled_gap, 'o-', color=color, label=f'L={L}', markersize=4)

    axes[0].axvline(1.0, color='red', linestyle='--', label=r'$\Gamma_c = J$')
    axes[0].set_xlabel(r'$\Gamma/J$')
    axes[0].set_ylabel(r'$L \cdot \Delta$')
    axes[0].set_title('Gap scaling: crossing identifies critical point')
    axes[0].legend(fontsize=9)

    # Order parameter profile at fixed g=0.5 for L=80
    L_op = 80
    model_op = TFIChain({'J': 1.0, 'g': 0.5, 'L': L_op, 'bc_MPS': 'finite'})
    psi_op = MPS.from_lat_product_state(model_op.lat, p_state=['up', 'down'])
    params_op = {
        'trunc_params': {'chi_max': chi_scan, 'svd_min': 1e-10},
        'mixer': True,
        'mixer_params': {'amplitude': 1e-4, 'decay': 1.5, 'disable_after': 30},
        'N_sweeps_check': 1, 'min_sweeps': 5, 'max_sweeps': 20,
    }
    _, psi_ordered = dmrg.TwoSiteDMRGEngine(psi_op, model_op, params_op).run()
    sigma_z = psi_ordered.expectation_value('Sigmaz')

    axes[1].plot(range(L_op), sigma_z, 'b-o', markersize=3)
    axes[1].axhline(0, color='gray', linestyle='--')
    axes[1].set_xlabel('Site $i$')
    axes[1].set_ylabel(r'$\langle \sigma^z_i \rangle$')
    axes[1].set_title(f'Order parameter profile, L={L_op}, g=0.5')

    plt.tight_layout()
    plt.show()
else:
    # ED-based demonstration at smaller sizes
    print("TeNPy not available — showing ED-based gap scaling at small sizes.")
    g_scan = np.linspace(0.5, 1.5, 11)
    sizes_ed = [8, 12, 16]
    fig, ax = plt.subplots(figsize=(8, 5))

    for N, color in zip(sizes_ed, ['tab:blue', 'tab:orange', 'tab:green']):
        scaled_gaps = []
        for g in g_scan:
            H = build_tfim_ed(N, J=1.0, Gamma=g)
            evals = eigsh(H, k=2, which='SA', return_eigenvectors=False)
            evals = np.sort(evals)
            scaled_gaps.append((evals[1] - evals[0]) * N)
        ax.plot(g_scan, scaled_gaps, 'o-', color=color, label=f'L={N}', markersize=4)

    ax.axvline(1.0, color='red', linestyle='--', label=r'$\Gamma_c = J$')
    ax.set_xlabel(r'$\Gamma/J$')
    ax.set_ylabel(r'$L \cdot \Delta$')
    ax.set_title('Gap scaling (ED, small L) — DMRG extends this to L~500')
    ax.legend()
    plt.tight_layout()
    plt.show()

## 6. Entanglement Entropy from DMRG

The entanglement entropy at each bond is directly available from the MPS singular values:

```python
EE = psi.entanglement_entropy()  # array of length N-1
```

This returns $S_A(\ell)$ for all cuts $\ell = 1, \ldots, N-1$. Comparing this profile across values of $\Gamma/J$ demonstrates the three regimes from lecture 07:

- **Gapped phases:** $S_A(\ell)$ saturates to a constant for large $\ell$.
- **Critical point (OBC Calabrese-Cardy):** $S_A(\ell) = \frac{1}{6} \log\!\left(\frac{2L}{\pi} \sin\frac{\pi \ell}{L}\right) + \text{const}$

Fitting the critical entropy profile at $L = 200$ gives a much cleaner central charge extraction ($c = 1/2$ for the Ising CFT) than the ED fit at $L = 20$ in lecture 07.

---

In [ ]:
def cc_formula(x, c_over_6, const):
    return c_over_6 * x + const


if TENPY_AVAILABLE:
    import logging
    logging.disable(logging.WARNING)

    L_ee = 100
    phases_ee = [(0.5, 'Ordered (g=0.5)', 'tab:blue'), (1.0, 'Critical (g=1.0)', 'tab:red'), (2.0, 'Disordered (g=2.0)', 'tab:green')]
    chi_ee = 64

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for g, label, color in phases_ee:
        model = TFIChain({'J': 1.0, 'g': g, 'L': L_ee, 'bc_MPS': 'finite'})
        psi = MPS.from_lat_product_state(model.lat, p_state=['up', 'down'])
        params = {
            'trunc_params': {'chi_max': chi_ee, 'svd_min': 1e-10},
            'mixer': True,
            'mixer_params': {'amplitude': 1e-5, 'decay': 1.5, 'disable_after': 30},
            'N_sweeps_check': 1, 'min_sweeps': 5, 'max_sweeps': 20,
        }
        _, psi_gs = dmrg.TwoSiteDMRGEngine(psi, model, params).run()
        ee = psi_gs.entanglement_entropy()

        axes[0].plot(range(1, L_ee), ee, '-', color=color, label=label, linewidth=1.5)

    axes[0].set_xlabel('Cut position $\\ell$')
    axes[0].set_ylabel(r'$S_A(\ell)$')
    axes[0].set_title(f'Entanglement entropy profile, L={L_ee}')
    axes[0].legend(fontsize=9)

    # Central charge fit at L=200
    L_cc = 200
    model_cc = TFIChain({'J': 1.0, 'g': 1.0, 'L': L_cc, 'bc_MPS': 'finite'})
    psi_cc = MPS.from_lat_product_state(model_cc.lat, p_state=['up', 'down'])
    params_cc = {
        'trunc_params': {'chi_max': 128, 'svd_min': 1e-10},
        'mixer': True,
        'mixer_params': {'amplitude': 1e-5, 'decay': 1.5, 'disable_after': 50},
        'N_sweeps_check': 1, 'min_sweeps': 5, 'max_sweeps': 20,
    }
    _, psi_cc_gs = dmrg.TwoSiteDMRGEngine(psi_cc, model_cc, params_cc).run()
    ee_cc = psi_cc_gs.entanglement_entropy()

    ells_cc = np.arange(1, L_cc)
    x_fit = np.log(2 * L_cc / np.pi * np.sin(np.pi * ells_cc / L_cc))
    popt, _ = curve_fit(cc_formula, x_fit, ee_cc)
    c_fit = popt[0] * 6

    axes[1].plot(x_fit, ee_cc, 'r.', markersize=3, label='DMRG data')
    axes[1].plot(x_fit, cc_formula(x_fit, *popt), 'k-', linewidth=2,
                 label=f'CC fit: c = {c_fit:.4f} (expected 0.5)')
    axes[1].set_xlabel(r'$\log(\frac{2L}{\pi}\sin\frac{\pi\ell}{L})$')
    axes[1].set_ylabel(r'$S_A(\ell)$')
    axes[1].set_title(f'Central charge extraction, L={L_cc}')
    axes[1].legend(fontsize=9)

    plt.tight_layout()
    plt.show()
    print(f"Fitted central charge c = {c_fit:.4f} (expected c = 0.5 for Ising CFT)")

else:
    # ED-based entropy + CC fit for comparison
    print("TeNPy not available — reproducing ED-based entropy fit from lecture 07.")
    from scipy.sparse.linalg import eigsh

    def entanglement_entropy(psi, ell, N):
        psi_mat = psi.reshape(2**ell, 2**(N - ell))
        rho = psi_mat @ psi_mat.conj().T
        eigvals = np.linalg.eigvalsh(rho)
        eigvals = eigvals[eigvals > 1e-15]
        return -np.sum(eigvals * np.log(eigvals))

    N_ed = 20
    H = build_tfim_ed(N_ed, J=1.0, Gamma=1.0)
    _, evecs = eigsh(H, k=1, which='SA')
    psi0 = evecs[:, 0]

    ells = np.arange(1, N_ed)
    S = np.array([entanglement_entropy(psi0, int(l), N_ed) for l in ells])
    x_fit = np.log(2 * N_ed / np.pi * np.sin(np.pi * ells / N_ed))
    popt, _ = curve_fit(cc_formula, x_fit, S)
    c_fit = popt[0] * 6

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.plot(x_fit, S, 'ro', markersize=4, label=f'ED, L={N_ed}')
    ax.plot(x_fit, cc_formula(x_fit, *popt), 'k-',
            label=f'CC fit: c = {c_fit:.3f} (expected 0.5)')
    ax.set_xlabel(r'$\log(\frac{2L}{\pi}\sin\frac{\pi\ell}{L})$')
    ax.set_ylabel(r'$S_A(\ell)$')
    ax.set_title(f'Central charge fit (ED, L={N_ed}) — DMRG gives cleaner result at L=200')
    ax.legend()
    plt.tight_layout()
    plt.show()
    print(f"ED central charge: c = {c_fit:.4f} (DMRG at L=200 would give c closer to 0.5)")

## 7. Pushing to Larger Systems

The real advantage of DMRG over ED becomes clear when you push to $L \sim 200$–$1000$.

**Finite-size scaling with large systems:** With $L$ up to 500, finite-size corrections are $\lesssim 1\%$ for most observables. Critical exponents extracted from scaling collapses at these sizes are far more reliable than those from ED at $L = 8$–20.

**Bond dimension convergence:** For the gapped TFIM away from criticality, $\chi = 32$ is sufficient. Near the critical point, $\chi = 128$ or $\chi = 256$ may be needed. The discarded weight is your convergence criterion.

**Computational time:** A DMRG run at $L = 500$, $\chi = 128$ takes seconds to minutes on a laptop. This is what makes DMRG a practical research tool — computations that once required supercomputers now run interactively.

---

In [ ]:
if TENPY_AVAILABLE:
    import logging, time
    logging.disable(logging.WARNING)

    # Scaling study: E0/L vs L at criticality
    # Exact thermodynamic limit: E0/L = -4/pi * J for the 1D TFIM at Gamma=J
    # (actually the exact value for the infinite chain with OBC is more subtle;
    #  we compare numerically)
    sizes_large = [20, 50, 100, 200, 500]
    chi_large = 64

    E_per_site = []
    timings = []

    for L in sizes_large:
        t0 = time.time()
        model = TFIChain({'J': 1.0, 'g': 1.0, 'L': L, 'bc_MPS': 'finite'})
        psi = MPS.from_lat_product_state(model.lat, p_state=['up', 'down'])
        params = {
            'trunc_params': {'chi_max': chi_large, 'svd_min': 1e-10},
            'mixer': True,
            'mixer_params': {'amplitude': 1e-5, 'decay': 1.5, 'disable_after': 50},
            'N_sweeps_check': 1, 'min_sweeps': 5, 'max_sweeps': 20,
        }
        E0, _ = dmrg.TwoSiteDMRGEngine(psi, model, params).run()
        dt = time.time() - t0
        E_per_site.append(E0 / L)
        timings.append(dt)
        print(f"L={L:>4}: E0/L = {E0/L:.8f}, time = {dt:.1f}s")

    # Thermodynamic limit estimate from FSS: E0/L = e_inf + a/L^2
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    inv_L2 = 1.0 / np.array(sizes_large)**2
    pfit = np.polyfit(inv_L2, E_per_site, 1)
    e_inf = pfit[1]

    axes[0].plot(inv_L2, E_per_site, 'ro', markersize=6, label='DMRG')
    x_ext = np.linspace(0, max(inv_L2) * 1.1, 100)
    axes[0].plot(x_ext, np.polyval(pfit, x_ext), 'k--', label=f'Extrapolated: e_∞ = {e_inf:.5f}')
    axes[0].set_xlabel(r'$1/L^2$')
    axes[0].set_ylabel(r'$E_0/L$')
    axes[0].set_title('Energy per site vs 1/L² at criticality')
    axes[0].legend(fontsize=9)

    axes[1].plot(sizes_large, timings, 'bs-', markersize=6)
    axes[1].set_xlabel('System size $L$')
    axes[1].set_ylabel('Wall time (s)')
    axes[1].set_title(f'DMRG runtime vs L ($\\chi={chi_large}$)')
    axes[1].set_xscale('log')
    axes[1].set_yscale('log')

    plt.tight_layout()
    plt.show()

else:
    # Show the finite-size convergence of E0/L from ED
    print("TeNPy not available — showing ED finite-size convergence at small L.")
    sizes_ed = [4, 6, 8, 10, 12, 14, 16, 18, 20]
    E_per_site_ed = []

    for N in sizes_ed:
        H = build_tfim_ed(N, J=1.0, Gamma=1.0)
        E0 = eigsh(H, k=1, which='SA', return_eigenvectors=False)[0]
        E_per_site_ed.append(E0 / N)
        print(f"N={N:>3}: E0/N = {E0/N:.8f}")

    inv_L2 = 1.0 / np.array(sizes_ed)**2
    pfit = np.polyfit(inv_L2, E_per_site_ed, 1)
    e_inf = pfit[1]

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.plot(inv_L2, E_per_site_ed, 'ro', markersize=6, label='ED')
    x_ext = np.linspace(0, max(inv_L2) * 1.1, 100)
    ax.plot(x_ext, np.polyval(pfit, x_ext), 'k--',
            label=f'Extrapolated: e_∞ = {e_inf:.5f}')
    ax.set_xlabel(r'$1/L^2$')
    ax.set_ylabel(r'$E_0/L$')
    ax.set_title('Finite-size convergence of E0/L (ED) — DMRG extends to L=500+')
    ax.legend()
    plt.tight_layout()
    plt.show()
    print(f"Extrapolated thermodynamic limit: e_inf = {e_inf:.6f}")
    print("Note: with DMRG at L=200-500, this extrapolation would be orders of magnitude more accurate.")